# PNAD Income Analysis Pipeline

This notebook is the single executable scientific analysis for the project. All numerical procedures are implemented in the documented `pnad_income` package; the notebook orchestrates the pipeline and displays the complete set of descriptive, distributional, and inequality outputs used to inspect the harmonized PNAD/PNAD Contínua income series.


## 1. Configuration and reproducible execution

The annual analytical records are read from `dados_refined/`. The package maps the stored Portuguese fields `ano` and `renda` to the canonical internal names `year` and `income` without altering the Parquet files. The complete available series is analyzed from 1976 to 2025.


In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd

from pnad_income.pipeline import PipelineConfig, pipeline_overview, run_pipeline
from pnad_income.plotting import (
    plot_ccdf,
    plot_ccdf_grid,
    plot_gini_evolution,
    plot_histogram,
    plot_histogram_grid,
    plot_lorenz_curve,
    plot_lorenz_grid,
    plot_measure_comparison,
    plot_measure_comparison_grid,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DATABASE_PATH = Path(os.environ.get("PNAD_DATABASE_PATH", "../dados_refined")).expanduser()
CONFIG = PipelineConfig(
    database_path=DATABASE_PATH,
    ccdf_base=1.05,
    start_year=1976,
    end_year=2025,
)
CONFIG


## 2. Database loading, validation, and analytical coverage

The pipeline loads all annual records, validates the longitudinal schema, attaches year-specific monetary metadata, and constructs the adjusted income series. The diagnostic table below reports the number of observations, temporal coverage, and CCDF output size before any interpretation of the results.


In [ ]:
results = run_pipeline(CONFIG)
overview = pipeline_overview(results)
display(overview)
display(results.panel.head())


## 3. Annual descriptive statistics

For every available survey year, the table reports the number of observations, positive support, arithmetic mean, median, standard deviation, and Gini coefficient for both nominal and adjusted income where available.


In [ ]:
summary = results.summary
display(summary)


## 4. Plot-selection interface

Every annual figure can be generated in two ways. Individual functions accept a single `year`, while grid functions accept an explicit list `years=[...]` and the desired `nrows` and `ncols`. If the requested list contains more years than `nrows × ncols`, additional pages are created automatically. Unused axes on the final page are hidden.

The cells below illustrate both interfaces. They are independent of the complete-series figures shown later.


In [ ]:
SELECTED_YEAR = 2025
SELECTED_YEARS = [1976, 1990, 2001, 2010, 2020, 2025]
GRID_NROWS = 2
GRID_NCOLS = 3

plot_histogram(
    results.panel,
    year=SELECTED_YEAR,
    value_col="income",
    bins=60,
    yscale="log",
)
plt.show()

for fig in plot_histogram_grid(
    results.panel,
    value_col="income",
    years=SELECTED_YEARS,
    bins=60,
    yscale="log",
    nrows=GRID_NROWS,
    ncols=GRID_NCOLS,
):
    plt.show()


## 5. Annual Gini coefficient

The following figure displays the Gini coefficient for every available survey year in a single temporal series. The `years` argument may be used to restrict the temporal interval or select non-contiguous years.


In [ ]:
plot_gini_evolution(summary, value_col="income")
plt.show()


## 6. Annual income histograms — linear frequency scale

All available survey years are displayed below as small multiples. The explicit grid is set to six rows and four columns, giving a 24-panel capacity per page.


In [ ]:
for fig in plot_histogram_grid(
    results.panel,
    value_col="income",
    years=results.years,
    bins=60,
    yscale="linear",
    nrows=6,
    ncols=4,
):
    plt.show()


## 7. Annual income histograms — logarithmic frequency scale

A logarithmic frequency axis reveals the low-frequency upper tail that is compressed in ordinary histograms.


In [ ]:
for fig in plot_histogram_grid(
    results.panel,
    value_col="income",
    years=results.years,
    bins=60,
    yscale="log",
    nrows=6,
    ncols=4,
):
    plt.show()


## 8. Complementary cumulative distribution function

For a nonnegative income variable \(X\), the empirical complementary cumulative distribution is

\[
\widehat{\overline F}(x)
=
\frac{1}{N}
\sum_{i=1}^{N}
\mathbf{1}(X_i\geq x).
\]

Geometric thresholds are defined over the strictly positive support, whereas finite zero-income observations remain in the denominator \(N\).


In [ ]:
ccdf = results.ccdf_nominal_adjusted
display(ccdf.head(20))


## 9. Individual annual distribution

Any year can be selected directly. The example below displays the 2025 CCDF on log-log axes. Change only `year` to obtain another survey wave.


In [ ]:
plot_ccdf(
    ccdf,
    year=2025,
    measure="income",
    transform="loglog",
)
plt.show()


## 10. Annual CCDFs — linear axes


In [ ]:
for fig in plot_ccdf_grid(
    ccdf,
    measure="income",
    years=results.years,
    transform="linear",
    nrows=6,
    ncols=4,
):
    plt.show()


## 11. Annual CCDFs — log-log axes


In [ ]:
for fig in plot_ccdf_grid(
    ccdf,
    measure="income",
    years=results.years,
    transform="loglog",
    nrows=6,
    ncols=4,
):
    plt.show()


## 12. Legacy \(\ln[\ln(\mathrm{CCDF})]\) diagnostic

The historical analysis also used the transformation \(\ln[\ln(\mathrm{CCDF}[\%])]\). It is retained as a legacy diagnostic and is distinct from the scale-invariant log-log representation.


In [ ]:
for fig in plot_ccdf_grid(
    ccdf,
    measure="income",
    years=results.years,
    transform="double_log",
    nrows=6,
    ncols=4,
):
    plt.show()


## 13. Annual Lorenz curves

The grid geometry is likewise selectable. The complete-series display below uses six rows and four columns per page.


In [ ]:
for fig in plot_lorenz_grid(
    results.panel,
    value_col="income",
    years=results.years,
    nrows=6,
    ncols=4,
):
    plt.show()


## 14. Individual Lorenz curve

A single survey year can be requested directly.


In [ ]:
plot_lorenz_curve(results.panel, year=2025, value_col="income")
plt.show()


## 15. Nominal versus adjusted distributions

The grid version compares the selected income measures year by year. The individual version below it can be used for a single year.


In [ ]:
for fig in plot_measure_comparison_grid(
    ccdf,
    measures=("income", "income_adj"),
    years=results.years,
    transform="loglog",
    nrows=6,
    ncols=4,
):
    plt.show()

plot_measure_comparison(
    ccdf,
    year=2025,
    measures=("income", "income_adj"),
    transform="loglog",
)
plt.show()


## 16. Effective-income availability

The current refined release contains the harmonized longitudinal income series only. If a future release includes a separate effective-income field, the pipeline can construct the corresponding annual distribution.


In [ ]:
ccdf_effective = results.ccdf_habitual_effective
if ccdf_effective.empty:
    print("The current refined database does not contain usable income_effective observations.")
else:
    display(ccdf_effective.head(20))


## 17. Final data-quality diagnostics

The final table reports missingness and numerical support for the central analytical measures.


In [ ]:
diagnostic_columns = [
    c for c in ("income", "income_adj", "income_effective", "income_effective_adj")
    if c in results.panel.columns
]
diagnostics = pd.DataFrame({
    "column": diagnostic_columns,
    "non_missing": [int(results.panel[c].notna().sum()) for c in diagnostic_columns],
    "missing": [int(results.panel[c].isna().sum()) for c in diagnostic_columns],
    "minimum": [results.panel[c].min() for c in diagnostic_columns],
    "maximum": [results.panel[c].max() for c in diagnostic_columns],
})
display(diagnostics)


## 18. Reusable result objects

The validated objects available after execution are `results.panel`, `results.summary`, `results.ccdf_nominal_adjusted`, and `results.ccdf_habitual_effective`. Subsequent analyses should consume these objects rather than recreate preprocessing or distributional calculations inside notebook cells.
